# DistilBERT Fine-Tuning on SQuAD v1.1 (Extractive Question Answering)

Run cells top to bottom. **Runtime > Change runtime type > GPU (T4)** before starting.

If you re-run the install cell after already importing `transformers`, restart the runtime (**Runtime > Restart session**) before continuing, or the newer `eval_strategy` / `processing_class` arguments below may fail.

**Note on the dataset:** the bare `squad` repo relies on a loading script, which newer `datasets` versions (4.0+) refuse to execute — same issue as the other notebooks. This uses `rajpurkar/squad`, the official parquet-converted mirror with identical `id`/`title`/`context`/`question`/`answers` fields.

**Note on evaluation:** QA can't score accuracy directly from raw logits the way classification does — you have to convert start/end logit positions back into text spans, then compare those spans against the ground-truth answers with the official SQuAD metric. The original script loaded that metric but never actually used it (`eval_strategy="no"`, no evaluation code). This notebook adds the missing postprocessing step and a real Exact Match / F1 score after training, following the standard Hugging Face QA postprocessing approach. Per-epoch evaluation is still skipped (`eval_strategy="no"`) because this postprocessing is too expensive to run every epoch — the real score is computed once at the end.

In [ ]:
!pip install -q -U transformers datasets evaluate accelerate

## Restart runtime here if this is not a fresh session
`Runtime > Restart session`, then continue from the next cell.

In [ ]:
import os
import collections
import numpy as np
import torch
import evaluate

from datasets import load_dataset

from transformers import (
    AutoTokenizer,
    AutoModelForQuestionAnswering,
    TrainingArguments,
    Trainer,
    set_seed,
)

set_seed(42)

## Environment check

In [ ]:
print("=" * 70)
print("ENVIRONMENT CHECK")
print("=" * 70)

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("CUDA version:", torch.version.cuda)
else:
    print("WARNING: GPU is not available.")
    print("Training will be significantly slower.")
    print("Go to Runtime > Change runtime type > GPU.")

print("=" * 70)

## Configuration

In [ ]:
MODEL_NAME = "distilbert-base-uncased"

DATASET_NAME = "rajpurkar/squad"

OUTPUT_DIR = "./qa_results"
FINAL_MODEL_DIR = "./best_distilbert_squad"

MAX_LENGTH = 384

DOC_STRIDE = 128

LEARNING_RATE = 3e-5

TRAIN_BATCH_SIZE = 8
EVAL_BATCH_SIZE = 8

NUM_EPOCHS = 2

WEIGHT_DECAY = 0.01

USE_FP16 = torch.cuda.is_available()

N_BEST_SIZE = 20
MAX_ANSWER_LENGTH = 30

# If you hit an out-of-memory error on a free-tier Colab GPU, lower this
# TRAIN_BATCH_SIZE = 4

## Load SQuAD dataset

In [ ]:
print("\n" + "=" * 70)
print("LOADING SQUAD DATASET")
print("=" * 70)

dataset = load_dataset(
    DATASET_NAME
)

print(dataset)

print("\nDataset sizes:")

for split in dataset:
    print(f"{split}: {len(dataset[split])}")

## Load tokenizer

In [ ]:
print("\n" + "=" * 70)
print("LOADING DISTILBERT TOKENIZER")
print("=" * 70)

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME
)

print("Tokenizer loaded successfully.")

## Preprocess training data

In [ ]:
print("\n" + "=" * 70)
print("PREPARING TRAINING DATA")
print("=" * 70)


def prepare_train_features(examples):

    questions = [
        question.strip()
        for question in examples["question"]
    ]

    inputs = tokenizer(
        questions,
        examples["context"],
        max_length=MAX_LENGTH,
        truncation="only_second",
        stride=DOC_STRIDE,
        return_overflowing_tokens=True,
        return_offsets_mapping=True,
        padding="max_length"
    )

    sample_mapping = inputs.pop(
        "overflow_to_sample_mapping"
    )

    offset_mapping = inputs.pop(
        "offset_mapping"
    )

    start_positions = []
    end_positions = []

    for i, offsets in enumerate(offset_mapping):

        input_ids = inputs["input_ids"][i]

        cls_index = input_ids.index(
            tokenizer.cls_token_id
        )

        sequence_ids = inputs.sequence_ids(i)

        sample_index = sample_mapping[i]

        answers = examples["answers"][sample_index]

        if len(answers["answer_start"]) == 0:

            start_positions.append(cls_index)
            end_positions.append(cls_index)

            continue

        start_char = answers["answer_start"][0]

        end_char = (
            start_char
            + len(answers["text"][0])
        )

        # Find the beginning and end of the context
        token_start_index = 0

        while (
            sequence_ids[token_start_index] != 1
        ):
            token_start_index += 1

        token_end_index = len(input_ids) - 1

        while (
            sequence_ids[token_end_index] != 1
        ):
            token_end_index -= 1

        # Answer is not completely inside this feature
        if (
            offsets[token_start_index][0] > start_char
            or offsets[token_end_index][1] < end_char
        ):

            start_positions.append(cls_index)
            end_positions.append(cls_index)

        else:

            # Find token containing answer start
            while (
                token_start_index < len(offsets)
                and offsets[token_start_index][0]
                <= start_char
            ):
                token_start_index += 1

            start_positions.append(
                token_start_index - 1
            )

            # Find token containing answer end
            while (
                token_end_index >= 0
                and offsets[token_end_index][1]
                >= end_char
            ):
                token_end_index -= 1

            end_positions.append(
                token_end_index + 1
            )

    inputs["start_positions"] = start_positions
    inputs["end_positions"] = end_positions

    return inputs


tokenized_train = dataset["train"].map(
    prepare_train_features,
    batched=True,
    remove_columns=dataset["train"].column_names
)

print("Training preprocessing completed.")

print(
    "Number of training features:",
    len(tokenized_train)
)

## Preprocess validation data
Keeps `example_id` and `offset_mapping` so answer spans can be mapped back to the original context text after prediction.

In [ ]:
print("\n" + "=" * 70)
print("PREPARING VALIDATION DATA")
print("=" * 70)


def prepare_validation_features(examples):

    questions = [
        question.strip()
        for question in examples["question"]
    ]

    inputs = tokenizer(
        questions,
        examples["context"],
        max_length=MAX_LENGTH,
        truncation="only_second",
        stride=DOC_STRIDE,
        return_overflowing_tokens=True,
        return_offsets_mapping=True,
        padding="max_length"
    )

    sample_mapping = inputs.pop(
        "overflow_to_sample_mapping"
    )

    example_ids = []

    for i in range(len(inputs["input_ids"])):

        sample_index = sample_mapping[i]

        example_ids.append(
            examples["id"][sample_index]
        )

        sequence_ids = inputs.sequence_ids(i)

        offsets = inputs["offset_mapping"][i]

        # Keep offsets only for context tokens
        inputs["offset_mapping"][i] = [
            offset if sequence_ids[k] == 1 else None
            for k, offset in enumerate(offsets)
        ]

    inputs["example_id"] = example_ids

    return inputs


tokenized_validation = dataset["validation"].map(
    prepare_validation_features,
    batched=True,
    remove_columns=dataset["validation"].column_names
)

print("Validation preprocessing completed.")

print(
    "Number of validation features:",
    len(tokenized_validation)
)

## Load DistilBERT QA model

In [ ]:
print("\n" + "=" * 70)
print("LOADING DISTILBERT QUESTION ANSWERING MODEL")
print("=" * 70)

model = AutoModelForQuestionAnswering.from_pretrained(
    MODEL_NAME
)

print("Model loaded successfully.")

## Load SQuAD evaluation metric

In [ ]:
print("\n" + "=" * 70)
print("SETTING UP SQUAD EVALUATION")
print("=" * 70)

metric = evaluate.load(
    "squad"
)

print("SQuAD metric loaded.")

## Training configuration

In [ ]:
print("\n" + "=" * 70)
print("TRAINING CONFIGURATION")
print("=" * 70)

training_args = TrainingArguments(

    output_dir=OUTPUT_DIR,

    eval_strategy="no",

    save_strategy="epoch",

    learning_rate=LEARNING_RATE,

    per_device_train_batch_size=TRAIN_BATCH_SIZE,

    per_device_eval_batch_size=EVAL_BATCH_SIZE,

    num_train_epochs=NUM_EPOCHS,

    weight_decay=WEIGHT_DECAY,

    fp16=USE_FP16,

    save_total_limit=2,

    logging_steps=500,

    report_to="none"
)

## Create trainer

In [ ]:
print("\n" + "=" * 70)
print("CREATING TRAINER")
print("=" * 70)

trainer = Trainer(

    model=model,

    args=training_args,

    train_dataset=tokenized_train,

    processing_class=tokenizer
)

print("Trainer ready.")

## Train model

In [ ]:
print("\n" + "=" * 70)
print("STARTING QUESTION ANSWERING FINE-TUNING")
print("=" * 70)

trainer.train()

print("\nTraining completed successfully!")

## Real SQuAD evaluation (Exact Match / F1)
Converts predicted start/end logits into answer text spans, then scores them with the official SQuAD metric. This is the missing piece from the original script.

In [ ]:
print("\n" + "=" * 70)
print("RUNNING FULL SQUAD EVALUATION")
print("=" * 70)


def postprocess_qa_predictions(
    examples,
    features,
    raw_predictions,
    n_best_size=N_BEST_SIZE,
    max_answer_length=MAX_ANSWER_LENGTH
):

    all_start_logits, all_end_logits = raw_predictions

    example_id_to_index = {
        k: i for i, k in enumerate(examples["id"])
    }

    features_per_example = collections.defaultdict(list)

    for i, feature_example_id in enumerate(features["example_id"]):
        features_per_example[
            example_id_to_index[feature_example_id]
        ].append(i)

    predictions = collections.OrderedDict()

    for example_index, example in enumerate(examples):

        feature_indices = features_per_example[example_index]

        context = example["context"]

        valid_answers = []

        for feature_index in feature_indices:

            start_logits = all_start_logits[feature_index]
            end_logits = all_end_logits[feature_index]

            offset_mapping = features[feature_index]["offset_mapping"]

            start_indexes = np.argsort(start_logits)[
                -1 : -n_best_size - 1 : -1
            ].tolist()

            end_indexes = np.argsort(end_logits)[
                -1 : -n_best_size - 1 : -1
            ].tolist()

            for start_index in start_indexes:

                for end_index in end_indexes:

                    if (
                        start_index >= len(offset_mapping)
                        or end_index >= len(offset_mapping)
                        or offset_mapping[start_index] is None
                        or offset_mapping[end_index] is None
                    ):
                        continue

                    if (
                        end_index < start_index
                        or end_index - start_index + 1 > max_answer_length
                    ):
                        continue

                    start_char = offset_mapping[start_index][0]
                    end_char = offset_mapping[end_index][1]

                    valid_answers.append({
                        "score": (
                            start_logits[start_index]
                            + end_logits[end_index]
                        ),
                        "text": context[start_char:end_char]
                    })

        if len(valid_answers) > 0:

            best_answer = sorted(
                valid_answers,
                key=lambda x: x["score"],
                reverse=True
            )[0]

        else:

            best_answer = {"text": "", "score": 0.0}

        predictions[example["id"]] = best_answer["text"]

    return predictions


# Keep a version of the validation features with example_id / offset_mapping
# for postprocessing, and a tensor-only version for the model itself.
validation_for_model = tokenized_validation.remove_columns(
    ["example_id", "offset_mapping"]
)

raw_predictions = trainer.predict(validation_for_model)

final_predictions = postprocess_qa_predictions(
    dataset["validation"],
    tokenized_validation,
    raw_predictions.predictions
)

formatted_predictions = [
    {"id": k, "prediction_text": v}
    for k, v in final_predictions.items()
]

references = [
    {"id": example["id"], "answers": example["answers"]}
    for example in dataset["validation"]
]

squad_results = metric.compute(
    predictions=formatted_predictions,
    references=references
)

print("\nSQuAD Evaluation Results:")

for key, value in squad_results.items():
    print(f"{key}: {value:.4f}")

## Save model

In [ ]:
print("\n" + "=" * 70)
print("SAVING QUESTION ANSWERING MODEL")
print("=" * 70)

os.makedirs(
    FINAL_MODEL_DIR,
    exist_ok=True
)

trainer.save_model(
    FINAL_MODEL_DIR
)

tokenizer.save_pretrained(
    FINAL_MODEL_DIR
)

print(
    "Model saved to:",
    os.path.abspath(FINAL_MODEL_DIR)
)

## Verify saved files

In [ ]:
print("\n" + "=" * 70)
print("SAVED MODEL FILES")
print("=" * 70)

for filename in sorted(
    os.listdir(FINAL_MODEL_DIR)
):

    filepath = os.path.join(
        FINAL_MODEL_DIR,
        filename
    )

    if os.path.isfile(filepath):

        size_mb = (
            os.path.getsize(filepath)
            / (1024 * 1024)
        )

        print(
            f"{filename:<40}"
            f"{size_mb:.2f} MB"
        )

## Reload model

In [ ]:
print("\n" + "=" * 70)
print("RELOADING SAVED MODEL")
print("=" * 70)

trained_tokenizer = AutoTokenizer.from_pretrained(
    FINAL_MODEL_DIR
)

trained_model = AutoModelForQuestionAnswering.from_pretrained(
    FINAL_MODEL_DIR
)

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

trained_model.to(device)

trained_model.eval()

print("Model successfully reloaded.")
print("Running on:", device)

## Question answering function

In [ ]:
def answer_question(
    question,
    context
):

    inputs = trained_tokenizer(
        question,
        context,
        return_tensors="pt",
        truncation="only_second",
        max_length=MAX_LENGTH,
        stride=DOC_STRIDE,
        return_offsets_mapping=True
    )

    offset_mapping = inputs.pop(
        "offset_mapping"
    )

    inputs = {
        key: value.to(device)
        for key, value in inputs.items()
    }

    with torch.no_grad():

        outputs = trained_model(
            **inputs
        )

    start_logits = outputs.start_logits[0]

    end_logits = outputs.end_logits[0]

    start_index = torch.argmax(
        start_logits
    ).item()

    end_index = torch.argmax(
        end_logits
    ).item()

    # Make sure the answer span is valid
    if end_index < start_index:

        end_index = start_index

    # Limit answer length
    if end_index - start_index + 1 > 30:

        end_index = start_index + 29

    answer_tokens = inputs[
        "input_ids"
    ][0][start_index:end_index + 1]

    answer = trained_tokenizer.decode(
        answer_tokens,
        skip_special_tokens=True
    )

    return answer

## Test question answering

In [ ]:
print("\n" + "=" * 70)
print("TESTING QUESTION ANSWERING MODEL")
print("=" * 70)


context = """
Artificial intelligence is a field of computer science that
focuses on creating systems capable of performing tasks that
normally require human intelligence. These tasks include
learning, reasoning, problem solving, understanding language,
and recognizing patterns. Machine learning is a major
subfield of artificial intelligence that allows computers
to learn from data without being explicitly programmed.
"""


question = "What is artificial intelligence?"


answer = answer_question(
    question,
    context
)


print("\nContext:")
print(context)

print("\nQuestion:")
print(question)

print("\nAnswer:")
print(answer)

## Second test

In [ ]:
print("\n" + "=" * 70)
print("SECOND TEST")
print("=" * 70)


context_2 = """
Python is a high-level programming language created by
Guido van Rossum. It was first released in 1991 and is
widely used for web development, data science, artificial
intelligence, automation, and scientific computing.
Python is known for its simple syntax and large ecosystem
of libraries.
"""


question_2 = "Who created Python?"


answer_2 = answer_question(
    question_2,
    context_2
)


print("\nContext:")
print(context_2)

print("\nQuestion:")
print(question_2)

print("\nAnswer:")
print(answer_2)

## Third test

In [ ]:
print("\n" + "=" * 70)
print("THIRD TEST")
print("=" * 70)


context_3 = """
The Transformer architecture was introduced in the 2017
research paper titled "Attention Is All You Need." The
architecture relies heavily on self-attention mechanisms
and became the foundation for many modern language models,
including BERT, GPT, and T5.
"""


question_3 = "When was the Transformer architecture introduced?"


answer_3 = answer_question(
    question_3,
    context_3
)


print("\nContext:")
print(context_3)

print("\nQuestion:")
print(question_3)

print("\nAnswer:")
print(answer_3)

## Summary

In [ ]:
print("\n" + "=" * 70)
print("QUESTION ANSWERING MODEL COMPLETE")
print("=" * 70)

print("Task: Extractive Question Answering")
print("Base Model:", MODEL_NAME)
print("Dataset: SQuAD v1.1")
print("Fine-tuned Model:", FINAL_MODEL_DIR)

print("\nStatus:")
print("Trained       ✓")
print("Evaluated     ✓")
print("Saved         ✓")
print("Reloaded      ✓")
print("Tested        ✓")

print("=" * 70)